In [16]:
from IPython.utils import io
import tqdm.notebook
import os, sys, random
total = 100
with tqdm.notebook.tqdm(total=total) as pbar:
    with io.capture_output() as captured:
        for i in range(total):
            pbar.update(1)
            print(f"Processing item {i+1}/{total}")


  0%|          | 0/100 [00:00<?, ?it/s]

In [17]:
#The code requires a column with the smiles strings and the header named as: Smiles

#pip install molvs
import pandas as pd
import rdkit
from rdkit import Chem
from rdkit.Chem import PandasTools

from rdkit.Chem import SanitizeMol
from rdkit.Chem import RemoveHs
from rdkit.Chem import AssignStereochemistry

from rdkit.Chem import inchi

#from molvs.standardize import Standardizer
from molvs.metal import MetalDisconnector
from molvs.fragment import LargestFragmentChooser
from molvs.normalize import Normalizer
from molvs.charge import Reionizer, Uncharger

from molvs import validate_smiles
#from molvs import standardize_smiles

In [18]:
from rdkit.Chem import PandasTools

# Compounds
df = pd.read_csv('/Users/francisco/Library/CloudStorage/OneDrive-Pessoal/Documentos/LabMol/DOUTORADO/SWE/BINDING_DB/bindingdb_raw_hsDHODH.tsv', sep='\t')
df = df[['Ligand SMILES', 'IC50 (nM)']]

initial_rows = len(df)
print(f"Initial rows: {initial_rows}")

# Step 1: remove missing IC50 values
before = len(df)
df = df.dropna(subset=['IC50 (nM)']).copy()
print(f"Step 1 - Removed missing IC50 values: {before - len(df)}")

# Step 2: remove IC50 values containing qualifier symbols (e.g., > or <)
before = len(df)
df = df[~df['IC50 (nM)'].astype(str).str.contains(r'[<>]', na=False)].copy()
print(f"Step 2 - Removed rows with qualifier symbols: {before - len(df)}")

# Step 3: convert IC50 to numeric and remove non-numeric values
before = len(df)
df['IC50 (nM)'] = pd.to_numeric(df['IC50 (nM)'], errors='coerce')
df = df.dropna(subset=['IC50 (nM)']).copy()
print(f"Step 3 - Removed non-numeric IC50 values: {before - len(df)}")

# Step 4: keep only IC50 values below 100 nM
before = len(df)
df = df[df['IC50 (nM)'] < 100].copy()
print(f"Step 4 - Removed IC50 values >= 100 nM: {before - len(df)}")

print(f"Final rows: {len(df)}")
df

Initial rows: 3658
Step 1 - Removed missing IC50 values: 260
Step 2 - Removed rows with qualifier symbols: 621
Step 3 - Removed non-numeric IC50 values: 0
Step 4 - Removed IC50 values >= 100 nM: 1270
Final rows: 1507


/var/folders/zw/yr2q39g93yj7zprvshxyj6840000gn/T/ipykernel_33758/1625536813.py:4: DtypeWarning: Columns (8,11,37,38) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/Users/francisco/Library/CloudStorage/OneDrive-Pessoal/Documentos/LabMol/DOUTORADO/SWE/BINDING_DB/bindingdb_raw_hsDHODH.tsv', sep='\t')


,Ligand SMILES,IC50 (nM)
0,CONC(=O)c1c(C)c(nc2ccc(F)cc12)-c1ccc(cc1)-c1cc...,0.120
7,C[C@H](Oc1cc(-c2cn(C)c(n2)C(C)(C)O)c(F)cc1C(=O...,0.179
8,C[C@H](Oc1cc(-c2cn(C)c(n2)C(C)(C)O)c(F)cc1C(=O...,0.179
9,CC(C)c1cn(-c2ccccc2C)c(=O)c2cc(F)c(cc12)-c1ccc...,0.180
10,C[C@H](Oc1cc(c(F)cc1C(=O)Nc1c(C)n[nH]c1Cl)-c1c...,0.181
...,...,...
3653,C[C@H](Oc1cc(c(F)cc1C(=O)Nc1c(C)cc(F)cc1Cl)-n1...,3.000
3654,C[C@H](Oc1cc(c(F)cc1C(=O)Nc1c(Cl)cccc1Cl)-n1nc...,3.000
3655,C[C@H](Oc1cc(c(F)cc1C(=O)Nc1c(F)cccc1F)-n1nc2C...,3.000
3656,C[C@H](Oc1cc(c(F)cc1C(=O)Nc1c(F)cccc1Cl)-n1nc2...,3.000


In [19]:
#NOTE: If is intended to determine the canonical tautomer it is important to take into account that the rdkit and molvs functions for this purpose
#sometimes don't preserve the stereochemistry, particularly it is adviced to use molvs over rdkit, since molvs presents this problem with less frequency.

def has_radicals(mol):
    return any(atom.GetNumRadicalElectrons() > 0 for atom in mol.GetAtoms())

def curation(x):  # Modified the arguments, as the initial call was with 1 argument.
    try:
        if not isinstance(x, str) or not x.strip():
            return "Error 1"

        # Parse and sanitize the input SMILES.
        mol = Chem.MolFromSmiles(x, sanitize=True)
        if mol is None:
            # If rdkit could not parse the smiles, returns Error 1
            return "Error 1"

        # Reject radical species (e.g., [O], [C]) for this curation workflow.
        if has_radicals(mol):
            return "Error 1"

        mol = Chem.RemoveHs(mol)                                  # Removal of explicit hydrogens.
        mol = MetalDisconnector().disconnect(mol)                 # Disconnects metal atoms covalently bonded to non-metals.
        mol = LargestFragmentChooser().choose(mol)                # Keeps only the largest fragment.

        allowed_elements = {"H", "B", "C", "N", "O", "F", "Si", "P", "S", "Cl", "Se", "Br", "I"}
        actual_elements = {atom.GetSymbol() for atom in mol.GetAtoms()}
        if actual_elements - allowed_elements:
            # If molecule contains other than the allowed elements, return Error 2
            return "Error 2"

        mol = Normalizer().normalize(mol)                         # Correct functional groups and recombine charges.
        mol = Reionizer().reionize(mol)                           # Ensure strongest acids protonate first.
        mol = Uncharger().uncharge(mol)                           # Attempts to neutralize the molecules.
        Chem.AssignStereochemistry(mol, force=True, cleanIt=True) # Recalculate stereochemistry.

        # Final sanitize before exporting to SMILES.
        Chem.SanitizeMol(mol)

        # Reject radical species generated during normalization/reionization.
        if has_radicals(mol):
            return "Error 1"

        curated_smiles = Chem.MolToSmiles(mol, isomericSmiles=True)

        # Round-trip validation: only return SMILES that RDKit can parse and sanitize again.
        mol_roundtrip = Chem.MolFromSmiles(curated_smiles, sanitize=True)
        if mol_roundtrip is None or has_radicals(mol_roundtrip):
            return "Error 3"

        return curated_smiles

    except Exception as e:
        print(f"Error processing SMILES: {x}. Error message: {e}")
        return "Something else was found"

In [20]:
df["SMILES_curated"] = [curation(smi) for smi in df["Ligand SMILES"]]
df

,Ligand SMILES,IC50 (nM),SMILES_curated
0,CONC(=O)c1c(C)c(nc2ccc(F)cc12)-c1ccc(cc1)-c1cc...,0.120,CONC(=O)c1c(C)c(-c2ccc(-c3ccccc3F)cc2)nc2ccc(F...
7,C[C@H](Oc1cc(-c2cn(C)c(n2)C(C)(C)O)c(F)cc1C(=O...,0.179,Cc1ccccc1NC(=O)c1cc(F)c(-c2cn(C)c(C(C)(C)O)n2)...
8,C[C@H](Oc1cc(-c2cn(C)c(n2)C(C)(C)O)c(F)cc1C(=O...,0.179,Cc1ccccc1NC(=O)c1cc(F)c(-c2cn(C)c(C(C)(C)O)n2)...
9,CC(C)c1cn(-c2ccccc2C)c(=O)c2cc(F)c(cc12)-c1ccc...,0.180,Cc1ccccc1-n1cc(C(C)C)c2cc(-c3ccc(C)c(N)n3)c(F)...
10,C[C@H](Oc1cc(c(F)cc1C(=O)Nc1c(C)n[nH]c1Cl)-c1c...,0.181,Cc1n[nH]c(Cl)c1NC(=O)c1cc(F)c(-c2ccc3c(n2)NCCC...
...,...,...,...
3653,C[C@H](Oc1cc(c(F)cc1C(=O)Nc1c(C)cc(F)cc1Cl)-n1...,3.000,Cc1cc(F)cc(Cl)c1NC(=O)c1cc(F)c(-n2nc3n(c2=O)CC...
3654,C[C@H](Oc1cc(c(F)cc1C(=O)Nc1c(Cl)cccc1Cl)-n1nc...,3.000,C[C@H](Oc1cc(-n2nc3n(c2=O)CCCOC3)c(F)cc1C(=O)N...
3655,C[C@H](Oc1cc(c(F)cc1C(=O)Nc1c(F)cccc1F)-n1nc2C...,3.000,C[C@H](Oc1cc(-n2nc3n(c2=O)CCCC3)c(F)cc1C(=O)Nc...
3656,C[C@H](Oc1cc(c(F)cc1C(=O)Nc1c(F)cccc1Cl)-n1nc2...,3.000,C[C@H](Oc1cc(-n2nc3n(c2=O)CCCOC3)c(F)cc1C(=O)N...


In [21]:
# Diagnostics: verify validity of curated SMILES
errors_summary = df["SMILES_curated"].value_counts().reindex(["Error 1", "Error 2", "Error 3", "Something else was found"], fill_value=0)

def is_valid_smiles(s):
    if not isinstance(s, str):
        return False
    m = Chem.MolFromSmiles(s, sanitize=True)
    if m is None:
        return False
    return not any(atom.GetNumRadicalElectrons() > 0 for atom in m.GetAtoms())

valid_mask = df["SMILES_curated"].apply(is_valid_smiles)
invalid_curated = int((~valid_mask).sum())
total_curated = int(valid_mask.shape[0])

print("Error summary:")
print(errors_summary)
print(f"Total curated entries: {total_curated}")
print(f"Invalid curated SMILES (RDKit parse/sanitize failed or radicals present): {invalid_curated}")

Error summary:
SMILES_curated
Error 1                     1
Error 2                     0
Error 3                     0
Something else was found    0
Name: count, dtype: int64
Total curated entries: 1507
Invalid curated SMILES (RDKit parse/sanitize failed or radicals present): 1


[10:57:31] SMILES Parse Error: syntax error while parsing: Error
[10:57:31] SMILES Parse Error: check for mistakes around position 1:
[10:57:31] Error
[10:57:31] ^
[10:57:31] SMILES Parse Error: Failed parsing SMILES 'Error' for input: 'Error'


In [22]:
# Delate smiles that rdkit could not read
print(df.shape)
df = df[df["SMILES_curated"] != "Error 1"]
print(df.shape)
# Delate smiles that no contain allowed atoms
df = df[df["SMILES_curated"] != "Error 2"]
print(df.shape)
# Delate other errors
df = df[df["SMILES_curated"] != "Something else was found"].reset_index(drop=True)
print(df.shape)

(1507, 3)
(1506, 3)
(1506, 3)
(1506, 3)


In [23]:
# Eliminate empty rows in the curated SMILES column
df = df.dropna(subset=["SMILES_curated"]).copy()

# Calculate and store InChI for each curated SMILES
def smiles_to_inchi(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        return inchi.MolToInchi(mol)
    except Exception:
        return None

df["InChI_curated"] = df["SMILES_curated"].apply(smiles_to_inchi)

# Remove rows where InChI could not be generated
df = df.dropna(subset=["InChI_curated"]).reset_index(drop=True)

# Delete duplicates based on InChI, keeping the first occurrence
df = df.drop_duplicates(subset=["InChI_curated"], keep="first").reset_index(drop=True)
print(df.shape)
df.head(10)

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefi

(1037, 4)


[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo

[10:57:31] WARNING: Omitted undefined stereo



,Ligand SMILES,IC50 (nM),SMILES_curated,InChI_curated
0,CONC(=O)c1c(C)c(nc2ccc(F)cc12)-c1ccc(cc1)-c1cc...,0.120,CONC(=O)c1c(C)c(-c2ccc(-c3ccccc3F)cc2)nc2ccc(F...,InChI=1S/C24H18F2N2O2/c1-14-22(24(29)28-30-2)1...
1,C[C@H](Oc1cc(-c2cn(C)c(n2)C(C)(C)O)c(F)cc1C(=O...,0.179,Cc1ccccc1NC(=O)c1cc(F)c(-c2cn(C)c(C(C)(C)O)n2)...,InChI=1S/C24H25F4N3O3/c1-13-8-6-7-9-18(13)29-2...
2,CC(C)c1cn(-c2ccccc2C)c(=O)c2cc(F)c(cc12)-c1ccc...,0.180,Cc1ccccc1-n1cc(C(C)C)c2cc(-c3ccc(C)c(N)n3)c(F)...,InChI=1S/C25H24FN3O/c1-14(2)20-13-29(23-8-6-5-...
3,C[C@H](Oc1cc(c(F)cc1C(=O)Nc1c(C)n[nH]c1Cl)-c1c...,0.181,Cc1n[nH]c(Cl)c1NC(=O)c1cc(F)c(-c2ccc3c(n2)NCCC...,InChI=1S/C22H20ClF4N5O2/c1-10-18(19(23)32-31-1...
4,C[C@H](Oc1cc(-c2cn(C)c(n2)C(C)(C)O)c(F)cc1C(=O...,0.190,C[C@H](Oc1cc(-c2cn(C)c(C(C)(C)O)n2)c(F)cc1C(=O...,"InChI=1S/C23H21ClF5N3O3/c1-11(23(27,28)29)35-1..."
5,CC(Oc1cc(c(F)cc1C(=O)Nc1c(C)n[nH]c1Cl)-c1ccc(F...,0.191,Cc1n[nH]c(Cl)c1NC(=O)c1cc(F)c(-c2ccc(F)c(CO)n2...,InChI=1S/C20H16ClF5N4O3/c1-8-17(18(21)30-29-8)...
6,C[C@H](Oc1cc(c(F)cc1C(=O)Nc1c(C)n[nH]c1C(F)(F)...,0.192,Cc1n[nH]c(C(F)(F)F)c1NC(=O)c1cc(F)c(-c2ccc(Cl)...,InChI=1S/C20H15ClF7N5O2/c1-7-15(16(33-32-7)20(...
7,CC(Oc1cc(c(F)cc1C(=O)Nc1c(C)n[nH]c1C(F)(F)F)-c...,0.192,Cc1n[nH]c(C(F)(F)F)c1NC(=O)c1cc(F)c(-c2ccc(Cl)...,InChI=1S/C20H15ClF7N5O2/c1-7-15(16(33-32-7)20(...
8,CC(O)c1nc(cn1C)-c1cc(O[C@@H](C)C(F)(F)F)c(cc1F...,0.193,Cc1n[nH]c(Cl)c1NC(=O)c1cc(F)c(-c2cn(C)c(C(C)O)...,InChI=1S/C20H20ClF4N5O3/c1-8-16(17(21)29-28-8)...
9,C[C@H](Oc1cc(c(F)cc1C(=O)Nc1c(F)cccc1Cl)-c1ccc...,0.195,Cc1ccc(-c2cc(O[C@@H](C)C(F)(F)F)c(C(=O)Nc3c(F)...,InChI=1S/C22H17ClF5N3O2/c1-10-6-7-17(30-20(10)...


In [24]:
#Add ID column
df.insert(0, 'ID_Control', [f"ID_{str(i+1).zfill(3)}" for i in range(len(df))])
df

,ID_Control,Ligand SMILES,IC50 (nM),SMILES_curated,InChI_curated
0,ID_001,CONC(=O)c1c(C)c(nc2ccc(F)cc12)-c1ccc(cc1)-c1cc...,0.120,CONC(=O)c1c(C)c(-c2ccc(-c3ccccc3F)cc2)nc2ccc(F...,InChI=1S/C24H18F2N2O2/c1-14-22(24(29)28-30-2)1...
1,ID_002,C[C@H](Oc1cc(-c2cn(C)c(n2)C(C)(C)O)c(F)cc1C(=O...,0.179,Cc1ccccc1NC(=O)c1cc(F)c(-c2cn(C)c(C(C)(C)O)n2)...,InChI=1S/C24H25F4N3O3/c1-13-8-6-7-9-18(13)29-2...
2,ID_003,CC(C)c1cn(-c2ccccc2C)c(=O)c2cc(F)c(cc12)-c1ccc...,0.180,Cc1ccccc1-n1cc(C(C)C)c2cc(-c3ccc(C)c(N)n3)c(F)...,InChI=1S/C25H24FN3O/c1-14(2)20-13-29(23-8-6-5-...
3,ID_004,C[C@H](Oc1cc(c(F)cc1C(=O)Nc1c(C)n[nH]c1Cl)-c1c...,0.181,Cc1n[nH]c(Cl)c1NC(=O)c1cc(F)c(-c2ccc3c(n2)NCCC...,InChI=1S/C22H20ClF4N5O2/c1-10-18(19(23)32-31-1...
4,ID_005,C[C@H](Oc1cc(-c2cn(C)c(n2)C(C)(C)O)c(F)cc1C(=O...,0.190,C[C@H](Oc1cc(-c2cn(C)c(C(C)(C)O)n2)c(F)cc1C(=O...,"InChI=1S/C23H21ClF5N3O3/c1-11(23(27,28)29)35-1..."
...,...,...,...,...,...
1032,ID_1033,Cc1cc(C(O)=O)c2n(C)c(nc2c1)-c1c(F)c(F)c(-c2ccc...,3.000,Cc1cc(C(=O)O)c2c(c1)nc(-c1c(F)c(F)c(-c3ccccc3)...,InChI=1S/C22H14F4N2O2/c1-10-8-12(22(29)30)20-1...
1033,ID_1034,OC(=O)c1cc(ccc1Nc1cnc(nc1)-c1ccccc1Cl)C1CC1,3.000,O=C(O)c1cc(C2CC2)ccc1Nc1cnc(-c2ccccc2Cl)nc1,InChI=1S/C20H16ClN3O2/c21-17-4-2-1-3-15(17)19-...
1034,ID_1035,C[C@H](Oc1cc(c(F)cc1C(=O)Nc1c(C)cc(F)cc1Cl)-n1...,3.000,Cc1cc(F)cc(Cl)c1NC(=O)c1cc(F)c(-n2nc3n(c2=O)CC...,InChI=1S/C23H20ClF5N4O3/c1-11-7-13(25)8-15(24)...
1035,ID_1036,C[C@H](Oc1cc(c(F)cc1C(=O)Nc1c(Cl)cccc1Cl)-n1nc...,3.000,C[C@H](Oc1cc(-n2nc3n(c2=O)CCCOC3)c(F)cc1C(=O)N...,"InChI=1S/C22H18Cl2F4N4O4/c1-11(22(26,27)28)36-..."


In [25]:
df = df[['ID_Control', 'SMILES_curated', 'IC50 (nM)']]
df

,ID_Control,SMILES_curated,IC50 (nM)
0,ID_001,CONC(=O)c1c(C)c(-c2ccc(-c3ccccc3F)cc2)nc2ccc(F...,0.120
1,ID_002,Cc1ccccc1NC(=O)c1cc(F)c(-c2cn(C)c(C(C)(C)O)n2)...,0.179
2,ID_003,Cc1ccccc1-n1cc(C(C)C)c2cc(-c3ccc(C)c(N)n3)c(F)...,0.180
3,ID_004,Cc1n[nH]c(Cl)c1NC(=O)c1cc(F)c(-c2ccc3c(n2)NCCC...,0.181
4,ID_005,C[C@H](Oc1cc(-c2cn(C)c(C(C)(C)O)n2)c(F)cc1C(=O...,0.190
...,...,...,...
1032,ID_1033,Cc1cc(C(=O)O)c2c(c1)nc(-c1c(F)c(F)c(-c3ccccc3)...,3.000
1033,ID_1034,O=C(O)c1cc(C2CC2)ccc1Nc1cnc(-c2ccccc2Cl)nc1,3.000
1034,ID_1035,Cc1cc(F)cc(Cl)c1NC(=O)c1cc(F)c(-n2nc3n(c2=O)CC...,3.000
1035,ID_1036,C[C@H](Oc1cc(-n2nc3n(c2=O)CCCOC3)c(F)cc1C(=O)N...,3.000


In [26]:
df.to_csv("DB_curada.csv", sep=";", index=False)